# Data analysis

This notebook retrieves experiment results and anlyses them in order to answer various research questions

In [2]:
from __future__ import division
import itertools

# import magni
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tabulate import tabulate

# Setup - the following path can be changed if the result file is in a
# different location. This should correspond to the `result_folder`
# variable in 'aspmci_reconstructions.py', line 488:
data_path = 'data/'
#additional_paths = ['/home/tha/ASPMCI-test/5665/0/aspmci_reconstructions_test/',
#                    '/home/tha/ASPMCI-test/5665/1/aspmci_reconstructions_test/']
hdf_database_name = '5699_merged_hdf_reconstruction_goblet_ID_.hdf5'
hdf_additional_names = ['5665_0_aspmci_reconstructions.hdf5',
                        '5665_1_aspmci_reconstructions.hdf5']

# Extract data for graphs/tables
# Load merged metrics database
hdf_stores = []
os.makedirs(data_path, exist_ok=True)

# should we obtain this data from somewhere?
hdf_store = pd.HDFStore(data_path + hdf_database_name)

reconstruction_metrics = hdf_store.select('/merged_metrics')
reconstruction_metrics.drop('array_job', axis=1, inplace=True)
assert len(reconstruction_metrics) == len(reconstruction_metrics.drop_duplicates())  # Assert no duplicate rows
hdf_stores.append(hdf_store)
reconstruction_metrics_add = pd.DataFrame()
for hdf_additional_name in hdf_additional_names:
    hdf_store = pd.HDFStore(data_path + hdf_additional_name)
    reconstruction_metrics_add = reconstruction_metrics_add.append(
        hdf_store.select('/simulation_results/metrics'),
        ignore_index=True)
    hdf_stores.append(hdf_store)
additional_metrics = reconstruction_metrics_add.query('reconstruction_algorithm == \'cubic_interpolation\'')
reconstruction_metrics = reconstruction_metrics.append(
    additional_metrics, ignore_index=True)

KeyError: 'No object named /merged_metrics in the file'

Find the worst image for all reconstruction algorithms

In [2]:
metric_grouper = reconstruction_metrics.groupby(['delta', 'image', 'reconstruction_algorithm', 'sampling_pattern'])
idx_best_of_reg_param = metric_grouper['psnr'].idxmax().values
best_of_reg_param = reconstruction_metrics.loc[idx_best_of_reg_param]
idx_worst_of_each_alg = best_of_reg_param.groupby(['reconstruction_algorithm'])['psnr'].idxmin().values
worst_of_each_alg = reconstruction_metrics.loc[idx_worst_of_each_alg.ravel()]
worst_of_each_alg

,delta,dictionary,image,psnr,reconstruction_algorithm,reconstruction_parameter,sampling_pattern,ssim,time
28596,0.175,NaN,image_2.mi,21.060110,cubic_interpolation,NaN,rect_spiral,0.704938,0.420253
9869,0.275,DCT,image_2.mi,10.657428,ell_1_amp,1.0000,uniform_lines,0.288440,5.103190
1951,0.100,DCT,image_1.mi,17.636571,ell_1_dct_overc2,3.0000,uniform_lines,0.273342,296.959248
16376,0.100,DCT,image_1.mi,17.834951,ell_1_dct_overc3,4.0000,uniform_lines,0.283993,1110.103961
23246,0.100,DCT,image_1.mi,21.619527,ell_1_dwt_db,3.0000,uniform_lines,0.719219,35.693890
16015,0.100,DCT,image_1.mi,22.348603,ell_1_dwt_dmey,4.0000,uniform_lines,0.747207,32.367794
16071,0.100,DCT,image_1.mi,22.409840,ell_1_dwt_sym,3.0000,uniform_lines,0.754349,27.002086
23005,0.125,DCT,image_1.mi,12.750716,ell_1_optim,0.1000,uniform_lines,0.235295,59.057464
487,0.150,DCT,image_0.mi,11.550876,iht_fixed,0.0500,uniform_lines,0.055073,4.362457
21907,0.100,DCT,image_0.mi,11.771878,ist_fixed,0.0500,uniform_lines,0.060422,6.455349


Find the worst image for all algorithms, for each sampling pattern

In [3]:
idx_worst_of_each_alg_patt = best_of_reg_param.groupby(['reconstruction_algorithm', 'sampling_pattern'])['psnr'].idxmin().values
worst_of_each_alg_patt = reconstruction_metrics.loc[idx_worst_of_each_alg_patt.ravel()]
worst_of_each_alg_patt

,delta,dictionary,image,psnr,reconstruction_algorithm,reconstruction_parameter,sampling_pattern,ssim,time
28596,0.175,NaN,image_2.mi,21.060110,cubic_interpolation,NaN,rect_spiral,0.704938,0.420253
28553,0.100,NaN,image_6.mi,32.856422,cubic_interpolation,NaN,uniform_lines,0.879641,0.310097
3259,0.300,DCT,image_3.mi,17.029488,ell_1_amp,1.000000e+00,rect_spiral,0.469836,5.204977
9869,0.275,DCT,image_2.mi,10.657428,ell_1_amp,1.000000e+00,uniform_lines,0.288440,5.103190
20766,0.100,DCT,image_6.mi,29.012462,ell_1_dct_overc2,8.000000e-01,rect_spiral,0.746394,327.041605
1951,0.100,DCT,image_1.mi,17.636571,ell_1_dct_overc2,3.000000e+00,uniform_lines,0.273342,296.959248
27943,0.100,DCT,image_6.mi,29.093146,ell_1_dct_overc3,8.000000e-01,rect_spiral,0.751795,887.319262
16376,0.100,DCT,image_1.mi,17.834951,ell_1_dct_overc3,4.000000e+00,uniform_lines,0.283993,1110.103961
8519,0.100,DCT,image_1.mi,23.842153,ell_1_dwt_db,8.000000e-01,rect_spiral,0.753811,100.875465
23246,0.100,DCT,image_1.mi,21.619527,ell_1_dwt_db,3.000000e+00,uniform_lines,0.719219,35.693890


What is the average difference in quality between TV and interpolation?

In [4]:
selection_interpolation_lines = best_of_reg_param.query('reconstruction_algorithm == "cubic_interpolation" and sampling_pattern == "uniform_lines"')
selection_tv_lines = best_of_reg_param.query('reconstruction_algorithm == "tv_optim" and sampling_pattern == "uniform_lines"')
difference_uniform_lines = (selection_interpolation_lines.loc[:,['psnr', 'ssim']].values - selection_tv_lines.loc[:,['psnr', 'ssim']].values).mean(axis=0)
difference_uniform_lines

array([ 0.63525128,  0.00753792])

In [5]:
selection_interpolation_spiral = best_of_reg_param.query('reconstruction_algorithm == "cubic_interpolation" and sampling_pattern == "rect_spiral"')
selection_tv_spiral = best_of_reg_param.query('reconstruction_algorithm == "tv_optim" and sampling_pattern == "rect_spiral"')
difference_rect_spiral = (selection_interpolation_spiral.loc[:,['psnr', 'ssim']].values - selection_tv_spiral.loc[:,['psnr', 'ssim']].values).mean(axis=0)
difference_rect_spiral

array([-2.54354837, -0.00967549])

What is the average difference in quality between spiral and raster patterns for interpolation?

In [6]:
difference_interpolation_patterns = (selection_interpolation_lines.loc[:,['psnr', 'ssim']].values - selection_interpolation_spiral.loc[:,['psnr', 'ssim']].values).mean(axis=0)
difference_interpolation_patterns

array([ 4.8951939 ,  0.02669997])

What is the average difference in quality between spiral and raster patterns for TV?

In [7]:
difference_tv_patterns = (selection_tv_lines.loc[:,['psnr', 'ssim']].values - selection_tv_spiral.loc[:,['psnr', 'ssim']].values).mean(axis=0)
difference_tv_patterns

array([ 1.71639425,  0.00948657])

How much worse than interpolation (`uniform_lines`) are Laplace AMP and $l_1$-minimisation (`rect_spiral`)?

In [8]:
selection_l1_amp_spiral = best_of_reg_param.query('reconstruction_algorithm == "ell_1_amp" and sampling_pattern == "rect_spiral"')
difference_l1_amp = (selection_interpolation_lines.loc[:,['psnr', 'ssim']].values - selection_l1_amp_spiral.loc[:,['psnr', 'ssim']].values).mean(axis=0)
difference_l1_amp

array([ 10.20931313,   0.12764509])

In [9]:
selection_ell1_spiral = best_of_reg_param.query('reconstruction_algorithm == "ell_1_optim" and sampling_pattern == "rect_spiral"')
difference_ell1 = (selection_interpolation_lines.loc[:,['psnr', 'ssim']].values - selection_ell1_spiral.loc[:,['psnr', 'ssim']].values).mean(axis=0)
difference_ell1

array([ 5.01902307,  0.05502027])

In [10]:
(selection_ell1_spiral.loc[:,['psnr', 'ssim']].values - selection_l1_amp_spiral.loc[:,['psnr', 'ssim']].values).mean(axis=0)

array([ 5.19029006,  0.07262482])

What is the average PSNR performance of some of the reconstruction algorithms?

In [11]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "ell_1_optim" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    36.433588
ssim     0.913389
dtype: float64

In [12]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "ell_1_dct_overc2" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    39.073001
ssim     0.931675
dtype: float64

In [13]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "ell_1_dct_overc3" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    39.097128
ssim     0.932290
dtype: float64

In [14]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "ell_1_dwt_db" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    31.644443
ssim     0.889632
dtype: float64

In [15]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "ell_1_dwt_dmey" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    32.756019
ssim     0.898814
dtype: float64

In [16]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "ell_1_dwt_sym" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    32.334362
ssim     0.899396
dtype: float64

In [17]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "tv_optim" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    39.100965
ssim     0.951385
dtype: float64

In [18]:
selection_ell1_lines = best_of_reg_param.query('reconstruction_algorithm == "cubic_interpolation" and sampling_pattern == "rect_spiral"')
selection_ell1_lines.loc[:,['psnr', 'ssim']].mean()

psnr    36.557417
ssim     0.941709
dtype: float64

In [19]:
selection_l1_amp_lines = best_of_reg_param.query('reconstruction_algorithm == "ell_1_amp" and sampling_pattern == "uniform_lines"')
selection_l1_amp_lines.loc[:,['psnr', 'ssim']].mean()

psnr    15.555086
ssim     0.360876
dtype: float64

In [20]:
selection_bg_amp_lines = best_of_reg_param.query('reconstruction_algorithm == "bg_amp" and sampling_pattern == "uniform_lines"')
selection_bg_amp_lines.loc[:,['psnr', 'ssim']].mean()

psnr   NaN
ssim   NaN
dtype: float64

In [21]:
selection_ist_lines = best_of_reg_param.query('reconstruction_algorithm == "ist_fixed" and sampling_pattern == "uniform_lines"')
selection_ist_lines.loc[:,['psnr', 'ssim']].mean()

psnr    16.574239
ssim     0.195066
dtype: float64

In [22]:
selection_iht_lines = best_of_reg_param.query('reconstruction_algorithm == "iht_fixed" and sampling_pattern == "uniform_lines"')
selection_iht_lines.loc[:,['psnr', 'ssim']].mean()

psnr    14.692893
ssim     0.162351
dtype: float64

Produce a table of the differences between the averages of all the PSNRs and SSIMs for different algorithms and patterns, averaged over under-sampling ratios and images.

In [23]:
psnrs = []
ssims = []
labels = []
pattern_names = {'rect_spiral': 'spiral', 'uniform_lines': 'raster'}
#recon_names = {'cubic_interpolation':'Interp.', 'tv_optim': 'TV', 'ell_1_optim': 'ell_1-min.', 'iht_fixed': 'IHT', 'ist_fixed': 'IST', 'ell_1_amp': 'Lapl. AMP', 'bg_amp': 'B.G. AMP'}
for pattern in best_of_reg_param['sampling_pattern'].unique():
    for algorithm in best_of_reg_param['reconstruction_algorithm'].unique():
        labels.append(algorithm)
        selection = best_of_reg_param.query('reconstruction_algorithm == @algorithm and sampling_pattern == @pattern')
        psnrs.append(selection['psnr'].mean())
        ssims.append(selection['ssim'].mean())
psnrs = np.array(psnrs).reshape((-1,1))
ssims = np.array(ssims).reshape((-1,1))
psnr_diffs = psnrs - psnrs.T
ssim_diffs = ssims - ssims.T
print tabulate(psnr_diffs, labels, tablefmt="latex", floatfmt='.1f')

\begin{tabular}{rrrrrrrrrrrrrrrrrrrrrr}
\hline
   cubic\_interpolation &   ell\_1\_amp &   ell\_1\_dct\_overc2 &   ell\_1\_dct\_overc3 &   ell\_1\_dwt\_db &   ell\_1\_dwt\_dmey &   ell\_1\_dwt\_sym &   ell\_1\_optim &   iht\_fixed &   ist\_fixed &   tv\_optim &   cubic\_interpolation &   ell\_1\_amp &   ell\_1\_dct\_overc2 &   ell\_1\_dct\_overc3 &   ell\_1\_dwt\_db &   ell\_1\_dwt\_dmey &   ell\_1\_dwt\_sym &   ell\_1\_optim &   iht\_fixed &   ist\_fixed &   tv\_optim \\
\hline
                   0.0 &         5.3 &               -2.5 &               -2.5 &            4.9 &              3.8 &             4.2 &           0.1 &        18.1 &         6.9 &       -2.5 &                  -4.9 &        21.0 &                5.2 &                4.6 &            5.5 &              3.1 &             3.4 &           8.6 &        21.9 &        20.0 &       -4.3 \\
                  -5.3 &         0.0 &               -7.8 &               -7.9 &           -0.4 &             -1.5 &            -1.1

Find the lowest-$\delta$ image for each algorithm that achieves SSIM > 0.9.

Configure `extract_recon_imgs.py` to extract the resulting images...

In [24]:
min_ssim = 0.9
for algorithm in best_of_reg_param['reconstruction_algorithm'].unique():
    selection = best_of_reg_param.query('reconstruction_algorithm == @algorithm and ssim >= 0.9 and image == "image_3.mi"')
    if not selection.empty:
        print(selection.loc[selection['delta'].idxmin()])

delta                                       0.1
dictionary                                  NaN
image                                image_3.mi
psnr                                   35.87309
reconstruction_algorithm    cubic_interpolation
reconstruction_parameter                    NaN
sampling_pattern                    rect_spiral
ssim                                  0.9671131
time                                  0.2187209
Name: 28489, dtype: object
delta                              0.15
dictionary                          DCT
image                        image_3.mi
psnr                           36.48528
reconstruction_algorithm      ell_1_amp
reconstruction_parameter              1
sampling_pattern            rect_spiral
ssim                          0.9432249
time                           7.825156
Name: 17493, dtype: object
delta                                    0.1
dictionary                               DCT
image                             image_3.mi
psnr               

In [25]:
for algorithm in best_of_reg_param['reconstruction_algorithm'].unique():
    selection = best_of_reg_param.query('reconstruction_algorithm == @algorithm and image == "image_0.mi"')
    if not selection.empty:
        print(selection.loc[selection['psnr'].idxmax()])

delta                                      0.25
dictionary                                  NaN
image                                image_0.mi
psnr                                   42.92871
reconstruction_algorithm    cubic_interpolation
reconstruction_parameter                    NaN
sampling_pattern                  uniform_lines
ssim                                  0.9797138
time                                   0.832623
Name: 28525, dtype: object
delta                             0.225
dictionary                          DCT
image                        image_0.mi
psnr                           35.68477
reconstruction_algorithm      ell_1_amp
reconstruction_parameter              1
sampling_pattern            rect_spiral
ssim                          0.9033815
time                           6.872186
Name: 7324, dtype: object
delta                                    0.3
dictionary                               DCT
image                             image_0.mi
psnr                

What is the average PSNR of cubic interpolation (best algorithm) at $\delta = 0.1$?

In [26]:
algorithm = 'cubic_interpolation'
selection = best_of_reg_param.query('reconstruction_algorithm == @algorithm and delta == 0.1 and sampling_pattern == "uniform_lines"')
selection['psnr'].mean()

35.76412382938748

How long did interpolation reconstruction (fastest) take on average?

In [27]:
algorithm = 'cubic_interpolation'
selection = best_of_reg_param.query('reconstruction_algorithm == @algorithm')
selection['time'].mean()

0.6003746191660563

In [28]:
for hdf_store in  hdf_stores:
    hdf_store.close()